# SANPO Bucket Streaming Kinetic Replay (Colab)

This notebook:
- mounts Google Drive once
- reads your processed JSONL session from Drive
- streams SANPO RGB frames directly from `gs://gresearch/sanpo_dataset/v0/sanpo-synthetic/`
- recomputes per-object kinetic scores using your custom formula
- writes an annotated MP4 back to Drive

No zip re-upload is required after the first Drive setup. Put the helper pack and your JSONL file in Drive, then open this notebook from Drive or Colab.

In [48]:
from __future__ import annotations

import json
import re
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import cv2
import numpy as np
from IPython.display import Video, display
from google.colab import drive


drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive')
PACK_DIR = DRIVE_ROOT / 'Capstone' / 'RandomColabStuff'
ZIP_PATH = PACK_DIR / 'sanpo_kinetic_replay_colab.zip'

if ZIP_PATH.exists() and not (PACK_DIR / 'sanpo_bucket_replay.py').exists():
    PACK_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Unzipping pack from {ZIP_PATH} to {PACK_DIR}')
    !unzip -o -q "{ZIP_PATH}" -d "{PACK_DIR}"

if str(PACK_DIR) not in sys.path:
    sys.path.insert(0, str(PACK_DIR))

from sanpo_bucket_replay import (
    DEFAULT_FORMULA,
    get_anonymous_client,
    list_sessions,
    parse_jsonl_session,
    render_replay_from_bucket,
)

print('Drive root:', DRIVE_ROOT)
print('Pack dir  :', PACK_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive root: /content/drive/MyDrive
Pack dir  : /content/drive/MyDrive/Capstone/RandomColabStuff


In [49]:
# ---------- User Config ----------
PROJECT_ROOT = PACK_DIR
SESSION_ID = 't0dGsEvxqyRwJj7Y12X2ZQzXxsD3l54u'

# Put your processed JSONL sessions in Google Drive, for example:
# /content/drive/MyDrive/Capstone/RandomColabStuff/sanpo_input/<session_file>.jsonl
JSONL_PATH = DRIVE_ROOT / 'Capstone' / 'RandomColabStuff' / 'sanpo_input' / f"{SESSION_ID}.jsonl"

# SANPO session folder name under data/sanpo/raw in Drive
# The notebook streams frame PNGs directly from the public bucket using this session id.


RGB_FRAMES_DIR = PROJECT_ROOT / f'data/sanpo/raw/{SESSION_ID}/camera_head/left/video_frames'
DESCRIPTION_JSON = PROJECT_ROOT / f'data/sanpo/raw/{SESSION_ID}/description.json'
OUTPUT_MP4 = DRIVE_ROOT / 'Capstone' / 'RandomColabStuff' / 'sanpo_output' / f'{JSONL_PATH.stem}_{SESSION_ID[:8]}_kinetic_replay.mp4'

# Render options
SCALE = 0.60
DRAW_TOP_N = 8
OUTPUT_FPS = None  # None => infer from session fps / frame stride

# ---------- Custom Kinetic Formula Parameters ----------
# K = severity(class) * direction_weight(position) * |velocity|^VELOCITY_EXP / max(distance, EPS_DISTANCE)^DISTANCE_EXP
EPS_DISTANCE = 0.5
VELOCITY_EXP = 2.0
DISTANCE_EXP = 1.0
UNKNOWN_DISTANCE_SCORE = 0.0
DEFAULT_SEVERITY = 1.0

AHEAD_WEIGHT = 1.00
LATERAL_WEIGHT = 0.85
FAR_LATERAL_WEIGHT = 0.70
REAR_WEIGHT = 0.60
UNKNOWN_DIRECTION_WEIGHT = 0.75

SEVERITY_BY_CLASS = {
    'person': 1.2,
    'bicycle': 1.4,
    'motorcycle': 1.8,
    'car': 2.0,
    'bus': 2.5,
    'truck': 2.5,
    'unlabeled_obstacle': 1.3,
    'fire hydrant': 0.8,
    'shadow': 0.2,
    'boat': 0.5,
}

print(f'JSONL:       {JSONL_PATH}')
print(f'SESSION_ID:  {SESSION_ID}')
print(f'RGB DIR:     {RGB_FRAMES_DIR}')
print(f'OUTPUT MP4:  {OUTPUT_MP4}')

JSONL:       /content/drive/MyDrive/Capstone/RandomColabStuff/sanpo_input/t0dGsEvxqyRwJj7Y12X2ZQzXxsD3l54u.jsonl
SESSION_ID:  t0dGsEvxqyRwJj7Y12X2ZQzXxsD3l54u
RGB DIR:     /content/drive/MyDrive/Capstone/RandomColabStuff/data/sanpo/raw/t0dGsEvxqyRwJj7Y12X2ZQzXxsD3l54u/camera_head/left/video_frames
OUTPUT MP4:  /content/drive/MyDrive/Capstone/RandomColabStuff/sanpo_output/t0dGsEvxqyRwJj7Y12X2ZQzXxsD3l54u_t0dGsEvx_kinetic_replay.mp4


In [50]:
@dataclass
class ObjectFact:
    object_id: str
    obj_class: str
    distance_m: Optional[float]
    velocity_ms: float
    position: str


OBJECT_PREFIX = re.compile(r'^(Object_\d+):\s*(.+)$')


def parse_distance(token: str) -> Optional[float]:
    token = token.strip().lower()
    m = re.search(r'([0-9]+(?:\.[0-9]+)?)m', token)
    return float(m.group(1)) if m else None


def parse_object_segment(segment: str) -> Optional[ObjectFact]:
    m = OBJECT_PREFIX.match(segment.strip())
    if not m:
        return None

    object_id = m.group(1)
    payload = m.group(2)
    parts = [p.strip() for p in payload.split(',')]
    if len(parts) < 2:
        return None

    obj_class = parts[0]
    distance_m = parse_distance(parts[1])

    vel_match = re.search(r'v=([-+]?\d*\.?\d+)m/s', payload)
    velocity_ms = float(vel_match.group(1)) if vel_match else 0.0

    pos_match = re.search(r'\),\s*([^\[]+)\s*\[', payload)
    position = pos_match.group(1).strip() if pos_match else 'unknown'

    return ObjectFact(
        object_id=object_id,
        obj_class=obj_class,
        distance_m=distance_m,
        velocity_ms=velocity_ms,
        position=position,
    )


def parse_jsonl_record(raw_line: str):
    row = json.loads(raw_line)
    assistant = json.loads(row.get('assistant', '{}'))
    if 'frame_id' not in assistant:
        return None

    frame_id = int(assistant['frame_id'])
    user_text = row.get('user', '')
    fact_text = user_text.split('] ', 1)[1] if '] ' in user_text else user_text

    object_segments = [
        seg.strip()
        for seg in fact_text.split('|')
        if seg.strip().startswith('Object_')
    ]

    objects = []
    for seg in object_segments:
        obj = parse_object_segment(seg)
        if obj is not None:
            objects.append(obj)

    return frame_id, objects


def direction_weight(position: str) -> float:
    p = position.lower()
    if 'ahead' in p:
        return AHEAD_WEIGHT
    if 'far-left' in p or 'far-right' in p:
        return FAR_LATERAL_WEIGHT
    if 'left' in p or 'right' in p:
        return LATERAL_WEIGHT
    if 'behind' in p:
        return REAR_WEIGHT
    return UNKNOWN_DIRECTION_WEIGHT


def kinetic_score_custom(obj: ObjectFact) -> float:
    if obj.distance_m is None:
        return float(UNKNOWN_DISTANCE_SCORE)

    severity = float(SEVERITY_BY_CLASS.get(obj.obj_class, DEFAULT_SEVERITY))
    v = abs(float(obj.velocity_ms))
    d = max(float(obj.distance_m), float(EPS_DISTANCE))

    return float(severity * direction_weight(obj.position) * (v ** VELOCITY_EXP) / (d ** DISTANCE_EXP))


def resolve_frame_path(frame_id: int) -> Optional[Path]:
    for ext in ('.png', '.jpg', '.jpeg'):
        p = RGB_FRAMES_DIR / f'{frame_id:06d}{ext}'
        if p.exists():
            return p
    return None


def infer_session_fps() -> Optional[float]:
    if not DESCRIPTION_JSON.exists():
        return None
    with open(DESCRIPTION_JSON, 'r', encoding='utf-8') as f:
        d = json.load(f)
    cams = d.get('session_camera_details', [])
    if not cams:
        return None
    fps = cams[0].get('fps')
    return float(fps) if fps else None


def infer_stride(frame_ids: list[int]) -> int:
    if len(frame_ids) < 2:
        return 1
    diffs = [b - a for a, b in zip(frame_ids[:-1], frame_ids[1:]) if b > a]
    if not diffs:
        return 1
    return max(1, int(round(float(np.median(np.array(diffs))))))

In [51]:
# -------- Config --------

BUCKET_NAME = 'gresearch'
BASE_PREFIX = 'sanpo_dataset/v0/sanpo-real'
CAMERA = 'camera_head'
SIDE = 'left'

CACHE_DIR = Path('/content/sanpo_frame_cache')
# Modified to save directly to Drive
OUTPUT_MP4 = PACK_DIR / 'sanpo_output' / f'replay_{SESSION_ID}_kinetic_replay.mp4'
# Ensure the output directory exists
Path(OUTPUT_MP4).parent.mkdir(parents=True, exist_ok=True)

DRAW_TOP_N = 8
SCALE = 0.60
OUTPUT_FPS = None  # None => infer from SANPO metadata and frame stride

# Custom kinetic formula (edit freely)
FORMULA = dict(DEFAULT_FORMULA)
FORMULA['velocity_exp'] = 2.0
FORMULA['distance_exp'] = 1.0
FORMULA['eps_distance'] = 0.5
FORMULA['unknown_distance_score'] = 0.0

# Direction weighting
FORMULA['ahead_weight'] = 1.00
FORMULA['lateral_weight'] = 0.85
FORMULA['far_lateral_weight'] = 0.70
FORMULA['rear_weight'] = 0.60
FORMULA['unknown_direction_weight'] = 0.75

# Class severity weighting
FORMULA['severity_by_class'] = {
    **DEFAULT_FORMULA['severity_by_class'],
    'unlabeled_obstacle': 1.4,
}

print('SESSION_ID =', SESSION_ID)
print('OUTPUT_MP4 =', OUTPUT_MP4)

SESSION_ID = t0dGsEvxqyRwJj7Y12X2ZQzXxsD3l54u
OUTPUT_MP4 = /content/drive/MyDrive/Capstone/RandomColabStuff/sanpo_output/replay_t0dGsEvxqyRwJj7Y12X2ZQzXxsD3l54u_kinetic_replay.mp4


In [52]:
frame_objects = parse_jsonl_session(JSONL_PATH)
frame_ids = sorted(frame_objects.keys())

all_objects = [o for objs in frame_objects.values() for o in objs]
unknown_dist = sum(1 for o in all_objects if o.distance_m is None)

print('Frames in JSONL:', len(frame_ids))
print('Frame id range :', frame_ids[0], '->', frame_ids[-1])
print('Object entries :', len(all_objects))
print('Unknown dist % :', round(100.0 * unknown_dist / max(1, len(all_objects)), 2))

Frames in JSONL: 142
Frame id range : 0 -> 423
Object entries : 881
Unknown dist % : 50.06


In [53]:
client = get_anonymous_client()
print('Anonymous SANPO client created.')

# Optional: show a few valid session ids from the bucket
print('Sample SANPO sessions:')
for sid in list_sessions(client, bucket_name=BUCKET_NAME, base_prefix=BASE_PREFIX, limit=5):
    print(' -', sid)

Anonymous SANPO client created.
Sample SANPO sessions:
 - -5OCPnbrwJdu3jH70ieU7pUiFsOJQoeG
 - -F9N8JhMuJZmOpz2M8o1At2j-jAas9AA
 - -MnIHGYpsPvn4CF_iAKGfRQewKSEZGv1
 - -PXxwecxoR8mWhFQ09j-lddcSMLO_0K2
 - -PqSDmiEe2pXjmYHgxh4YEBsj0T5LU10


In [54]:
summary = render_replay_from_bucket(
    client=client,
    session_id=SESSION_ID,
    frame_objects=frame_objects,
    output_mp4=OUTPUT_MP4,
    formula_cfg=FORMULA,
    cache_dir=CACHE_DIR,
    draw_top_n=DRAW_TOP_N,
    scale=SCALE,
    output_fps=OUTPUT_FPS,
    bucket_name=BUCKET_NAME,
    base_prefix=BASE_PREFIX,
    camera=CAMERA,
    side=SIDE,
)

print(summary)

Rendered 1/142 frames
Rendered 31/142 frames
Rendered 61/142 frames
Rendered 91/142 frames
Rendered 121/142 frames
{'output_mp4': '/content/drive/MyDrive/Capstone/RandomColabStuff/sanpo_output/replay_t0dGsEvxqyRwJj7Y12X2ZQzXxsD3l54u_kinetic_replay.mp4', 'rendered_frames': 142, 'missing_frames': 0, 'total_frame_ids': 142, 'fps': 5.0, 'stride': 3, 'session_fps': 15.0}


In [55]:
# display(Video(str(OUTPUT_MP4), embed=True, html_attributes='controls loop'))